# Mosaicking ITS\_LIVE granules into a virtual cube

This notebook virtualizes two [ITS\_LIVE](https://its-live.jpl.nasa.gov/) glacier-velocity
granules that share a global grid but each cover only part of it, then uses native xarray
`concat(..., join="outer")` to align them onto the shared grid (sparse, with fill where a
granule has no data) and stack them along `time`. The data variables stay virtual
(`ManifestArray`) throughout — only the coordinates are loaded — and the result is written to
Icechunk without copying any pixel data.


In [ ]:
import os

import numpy as np
import obstore
import xarray as xr
from obspec_utils.registry import ObjectStoreRegistry

import virtualizarr as vz
from virtualizarr.parsers import HDFParser

In [ ]:
bucket = "s3://its-live-data"
key = "test-space/virtual-cubes"

In [ ]:
granule_1 = "LC08_L1GT_020121_20231013_20231102_02_T2_X_LC09_L1GT_020121_20231106_20231106_02_T2_G0120V02_P084.nc"

In [ ]:
store = obstore.store.from_url(bucket, region="us-west-2", skip_signature=True)
registry = ObjectStoreRegistry({bucket: store})
parser = HDFParser(drop_variables=["mapping", "img_pair_info"])

In [ ]:
vds_1 = vz.open_virtual_dataset(
    url=os.path.join(bucket, key, granule_1),
    parser=parser,
    registry=registry,
    # load x/y/time as real, indexed coordinates so xarray has an index on every
    # dimension for alignment. Data variables stay virtual (ManifestArray).
    loadable_variables=["time", "y", "x"],
    decode_times=True,
)
vds_1

In [ ]:
granule_2 = "LC08_L1GT_020120_20201121_20210315_02_T2_X_LC08_L1GT_020120_20210124_20210305_02_T2_G0120V02_P051.nc"

In [ ]:
vds_2 = vz.open_virtual_dataset(
    url=os.path.join(bucket, key, granule_2),
    parser=parser,
    registry=registry,
    loadable_variables=["time", "y", "x"],
    decode_times=True,
)
vds_2

In [ ]:
# The ITS_LIVE data uses a descending y coordinate.  Internally, xarray's alignment machinery requires ascending
# ordering so the y coordinate needs to be flipped prior to concatenation.
flip = lambda ds: ds.assign_coords(y=-ds.y)  # descending y  ->  ascending -y
flip_1 = flip(vds_1)
flip_2 = flip(vds_2)
cube = xr.concat(
    [flip_1, flip_2], dim="time", join="outer", combine_attrs="drop_conflicts"
)
cube = flip(cube)
cube

In [ ]:
import shutil

import icechunk as ic

# the referenced chunk data lives on this public, anonymous S3 bucket
url_prefix = "s3://its-live-data/"
store_path = "its_live_cube.icechunk"

# start fresh so this cell is re-runnable
shutil.rmtree(store_path, ignore_errors=True)

# register a virtual chunk container so icechunk knows how to read the s3 refs
config = ic.RepositoryConfig.default()
config.set_virtual_chunk_container(
    ic.VirtualChunkContainer(
        url_prefix, ic.s3_store(region="us-west-2", anonymous=True)
    )
)
repo = ic.Repository.create(
    storage=ic.local_filesystem_storage(store_path),
    config=config,
    authorize_virtual_chunk_access=ic.containers_credentials(
        {url_prefix: ic.s3_credentials(anonymous=True)}
    ),
)


# icechunk metadata is strict JSON and cannot represent NaN/inf, but ITS_LIVE
# stores NaN in some attributes (e.g. 'stable_shift_stationary'); drop those.
def _drop_nonfinite_attrs(ds):
    ds = ds.copy()

    def clean(attrs):
        keep = {}
        for k, v in attrs.items():
            arr = np.asarray(v)
            if arr.dtype.kind == "f" and not np.isfinite(arr).all():
                continue
            keep[k] = v
        return keep

    ds.attrs = clean(ds.attrs)
    for var in ds.variables.values():
        var.attrs = clean(var.attrs)
    return ds


# write the cube: only the virtual references are stored, no pixel data is copied
session = repo.writable_session("main")
_drop_nonfinite_attrs(cube).vz.to_icechunk(session.store)
snapshot_id = session.commit(
    "its_live virtual cube: 2 granules mosaicked + stacked on time"
)
print("committed snapshot", snapshot_id)

# reopen from the committed store
cube_roundtrip = xr.open_zarr(
    repo.readonly_session("main").store, consolidated=False, zarr_format=3
)
cube_roundtrip

In [ ]:
v = cube_roundtrip["v"]
dv = v.isel(time=1) - v.isel(time=0)
dv

In [ ]:
import matplotlib.pyplot as plt

# surface speed `v` at each time step, on a shared color scale
v = cube_roundtrip["v"].load()
vmax = float(np.nanpercentile(v, 98))

nt = v.sizes["time"]
fig, axes = plt.subplots(1, nt, figsize=(7 * nt, 6), constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, t in zip(axes, range(nt)):
    im = v.isel(time=t).plot.imshow(
        ax=ax, cmap="viridis", vmin=0, vmax=vmax, add_colorbar=False
    )
    ax.set_aspect("equal")
    ax.set_title(str(v["time"].values[t])[:10])
fig.colorbar(im, ax=list(axes), label="speed (m/yr)", shrink=0.8)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

lim = np.nanpercentile(np.abs(dv), 98)

# plot
fig, ax = plt.subplots(figsize=(9, 7))
dv.plot.imshow(
    ax=ax,
    cmap="RdBu_r",  # red = speedup, blue = slowdown
    vmin=-lim,
    vmax=lim,  # symmetric, centered on zero
    cbar_kwargs={"label": "Δ speed (m/yr)"},
)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()